# EMA + RSI

## Table of contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [EMA + RSI](#ema-rsi-section)
  - [Backtesting](#backtesting)
  - [Parameters sweep](#parameters-sweep)
  - [Walk-forward analysis](#walk-forward-analysis)
  - [Monte Carlo simulations](#monte-carlo-simulations)
- [Inverse EMA + RSI](#inverse-ema--rsi)
  - [Backtesting](#inv-backtesting)
  - [Parameters sweep](#inv-parameters-sweep)
  - [Walk-forward analysis](#inv-walk-forward)
  - [Monte Carlo simulations](#inv-monte-carlo)

EMA Crossover + RSI Filter \
A momentum strategy that trades in the direction of the cross: fast EMA(9) over slow EMA(21) \
It uses ATR(14) for a dynamic volatility-adaptive trailing stop. \
Filters false signals with RSI(14) to cut whipsaws in ranging markets. \
Works across timeframes; well suited to scalping and intraday.

__How EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover (signal flip) OR price hits the ATR-based trailing stop.
- The RSI filter reduces whipsaws in ranging markets.

__The RSI filter__ on ema / ema_inv is an optional, configurable filter:
- filter off entirely: StrategyConfig(rsi_filter=False)
- custom bounds: StrategyConfig(rsi_bullish=65.0, rsi_bearish=35.0)

## Configuration

### Setup

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result
from engine.strategy_configurator import StrategyConfig, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import sweep, walk_forward, monte_carlo

import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = StrategyConfig()    # engine/strategy_configurator.py (signal knobs)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic StrategyConfig().
STRATEGY_OVERRIDES = {}      # e.g. {"ema_fast": 12, "ema_slow": 26, "rsi_filter": False}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [ ]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

<a id="ema-rsi-section"></a>
## EMA + RSI

### Backtesting

In [5]:
# Import EMA + RSI strategy
from engine.strategies import EMACrossoverStrategy
STRATEGY = EMACrossoverStrategy

In [ ]:
# Backtest EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

### Parameters sweep

In [ ]:
# One full-history grid sweep.
# It is not feeding the walk-forward.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2          # ignore degenerate combos with too few trades to be meaningful
sw = sweep(STRATEGY, df, GRID, symbol=SYMBOL, interval=INTERVAL,
           trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY)
sw[sw.trades >= MIN_TRADES].sort_values("total_pnl_bps", ascending=False).head(8)

In [ ]:
# Look at the region of good parameters, not a single peak:
# a bright cell with bright neighbours is robust,
# a lone bright cell is usually an overfit outlier.
px.imshow(
    sw.pivot(index="ema_fast", columns="ema_slow", values="sharpe_approx"),
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0, aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Sharpe"),
    title=f"In-sample Sharpe — {strategy.name} | {SYMBOL} {INTERVAL}m (single window)",
).show()

### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the full grid every train window.
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample and their out-of-sample performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold.
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window
wf.param_stability()

In [ ]:
# Out-of-sample equity: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.
eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
px.line(eq, labels={"value": "equity", "index": ""},
        title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m").update_layout(showlegend=False).show()

### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.
px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"},
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()
# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"},
             title="OOS max-drawdown distribution").show()

## Inverse EMA + RSI

Inverse EMA Crossover + RSI Filter \
Mean-reversion counterpart to EMA+RSI: fades the cross instead of riding it. \
Uses ATR(14) for dynamic trailing stops (adapts to volatility) and RSI(14) filter. \
Bets that EMA crosses mark momentum exhaustion, not continuation.

__How Inverse EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) — standard for crypto.
- Short Entry: Fast EMA crosses above Slow EMA AND RSI(14) > 30 (not oversold) — fading the bullish cross.
- Long Entry:  Fast EMA crosses below Slow EMA AND RSI(14) < 70 (not overbought) — fading the bearish cross.
- Exit: Opposite crossover (signal flip) OR price hits ATR-based trailing stop.
- Works best in range-bound / mean-reverting regimes; likely underperforms in strong trends.


<a id="inv-backtesting"></a>
### Backtesting

In [ ]:
# Import Inverse EMA + RSI strategy
from engine.strategies import InverseEMACrossoverStrategy
STRATEGY = InverseEMACrossoverStrategy

In [ ]:
# Backtest Inverse EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Inverse EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="inv-parameters-sweep"></a>
### Parameters sweep

In [ ]:
# One full-history grid sweep.
# It is not feeding the walk-forward.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2          # ignore degenerate combos with too few trades to be meaningful
sw = sweep(STRATEGY, df, GRID, symbol=SYMBOL, interval=INTERVAL,
           trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY)
sw[sw.trades >= MIN_TRADES].sort_values("total_pnl_bps", ascending=False).head(8)

In [ ]:
# Look at the region of good parameters, not a single peak:
# a bright cell with bright neighbours is robust,
# a lone bright cell is usually an overfit outlier.
px.imshow(
    sw.pivot(index="ema_fast", columns="ema_slow", values="sharpe_approx"),
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0, aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Sharpe"),
    title=f"In-sample Sharpe — {strategy.name} | {SYMBOL} {INTERVAL}m (single window)",
).show()

<a id="inv-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the full grid every train window.
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample and their out-of-sample performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold. Stable across folds = trustworthy; jumpy = overfit-prone.
wf.param_stability()

In [ ]:
# Out-of-sample equity: the stitched test windows compounded — the path you'd have
# lived through re-tuning periodically on only past data.
eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
px.line(eq, labels={"value": "equity", "index": ""},
        title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m").update_layout(showlegend=False).show()

<a id="inv-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# Monte Carlo. The walk-forward gave us ONE ordering of the out-of-sample trades.
# Here we shuffle those trades thousands of times — in small contiguous chunks, so
# losing streaks stay realistic — and rebuild the equity curve each time. The spread
# shows the range of outcomes luck alone could have produced, not just the one we got.
mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.
px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"},
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()
# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"},
             title="OOS max-drawdown distribution").show()